# 2. Load a platform run and run inference locally

Weights and the ONNX model come from a run on the platform through the Metaflow Client API. Nothing is unpickled:
the weights are plain safetensors bytes and the ONNX file is self-contained.

The newest successful run is used by default. To pick one, run `pixi run load-local TrainCanDetectorFlow/<run id>`, or edit the `pathspec = ...` line in the cell below.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Image

from candetect.config import PRODUCTION_TAG, TRAIN_FLOW
from candetect.data import DataConfig, generate_dataset, make_splits
from candetect.inference import contact_sheet, latest_successful_run, load_run
from candetect.model import model_from_bytes
from candetect.train import confusion_matrix, metrics_from_confusion, predict_proba

OUT = Path("../artifacts/local")
OUT.mkdir(parents=True, exist_ok=True)

## Pick and load a run

In [ ]:
pathspec = os.environ.get("CANDETECT_RUN") or latest_successful_run()
if "/" not in pathspec:
    pathspec = f"{TRAIN_FLOW}/{pathspec}"
run = load_run(pathspec)
print("run:          ", run.pathspec)
print("model version:", run.model_version)
print("architecture: ", run.arch)
print("dataset hash: ", run.dataset_hash)
print("production:   ", PRODUCTION_TAG in run.tags)
print("tags:         ", run.tags)
print("recorded test metrics:", {k: round(v, 4) for k, v in run.metrics.items() if k != "n"})

## Same data version?

Regenerate the dataset locally from the run's recorded config. The hash must equal the one the platform recorded.

In [ ]:
data = generate_dataset(DataConfig.from_dict(run.data_config))
splits = make_splits(data.labels, seed=data.config.seed)
print("local hash:   ", data.sha256)
print("platform hash:", run.dataset_hash)
assert data.sha256 == run.dataset_hash, "local data differs from the data this model was trained on"
print("identical data version")

## Inference with onnxruntime, cross-checked against PyTorch

In [ ]:
test_images, test_labels = data.images[splits.test], data.labels[splits.test]

onnx_probs = run.onnx_model().predict_proba(test_images)
torch_probs = predict_proba(model_from_bytes(run.weights), test_images)
print("max |onnx - torch| probability difference:", float(np.abs(onnx_probs - torch_probs).max()))

cm = confusion_matrix(test_labels, onnx_probs.argmax(1))
metrics = metrics_from_confusion(cm)
print("local metrics:   ", {k: round(v, 4) for k, v in metrics.items() if k != "n"})
print("recorded metrics:", {k: round(v, 4) for k, v in run.metrics.items() if k != "n"})
assert abs(metrics["f1"] - run.metrics["f1"]) < 1e-3
pd.DataFrame(cm, index=["true normal", "true fallen_can"], columns=["pred normal", "pred fallen_can"])

## A few predictions

Green border: correct. Red border: wrong. `T` is the true class, `P` the prediction with its probability.

In [ ]:
png = contact_sheet(test_images[:8], test_labels[:8], onnx_probs[:8])
(OUT / "predictions.png").write_bytes(png)
Image(data=png)

In [ ]:
(OUT / "from_platform.onnx").write_bytes(run.onnx)
print("saved", OUT / "from_platform.onnx", f"({len(run.onnx) / 1e6:.1f} MB)")

Next: compare runs and promote one, `pixi run compare` and `pixi run promote <run id>`.